# Practice 2 – Hands-on Practice with Pre-trained Neural Network Architectures

## 1. Giới thiệu

Bài thực hành này tập trung vào việc sử dụng các mô hình mạng nơ-ron đã được **tiền huấn luyện (pre-trained)** từ `torchvision.models` để thực hiện bài toán phân loại ảnh thông qua kỹ thuật **Transfer Learning**.

Thay vì xây dựng và huấn luyện một mạng neural network từ đầu như Practice 1, Transfer Learning tận dụng các trọng số đã được học từ tập dữ liệu ImageNet (1.2 triệu ảnh, 1000 lớp) của các mô hình lớn. Nhờ đó, mô hình có thể đạt hiệu năng cao hơn với ít dữ liệu và thời gian huấn luyện hơn.

Bộ dữ liệu được sử dụng vẫn là **FashionMNIST** — giống Practice 1 — nhằm dễ dàng so sánh hiệu năng giữa hai cách tiếp cận.

**Quy trình thực hiện bao gồm:**

 1. Giới thiệu
 2. Import thư viện
 3. Tải và tiền xử lý dữ liệu
 4. Khám phá kiến trúc mô hình tiền huấn luyện
 5. Thích ứng mô hình cho bài toán cụ thể (Adapt Model)
 6. Thiết lập TensorBoard
 7. Thiết lập Loss Function và Optimizer
 8. Huấn luyện mô hình (Feature Extraction)
 9. Trực quan hóa kết quả huấn luyện
 10. Đánh giá mô hình
 11. Hiển thị dự đoán và nhãn thực tế
 12. Thử nghiệm kiến trúc và Hyperparameters
 13. So sánh kết quả thực nghiệm
 14. Lưu và tải mô hình
 15. Kết luận

## 2. Import Libraries

Import các thư viện cần thiết:
- `torch`, `torchvision`: PyTorch và các tiện ích ảnh, bao gồm `torchvision.models` cung cấp sẵn các mô hình tiền huấn luyện.
- `torch.utils.tensorboard`: Ghi log để theo dõi quá trình huấn luyện trực quan trên TensorBoard.
- `matplotlib`, `pandas`: Trực quan hóa dữ liệu, biểu đồ và bảng tổng hợp kết quả.

In [ ]:
import os
import copy
import time

import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

print(f"PyTorch version  : {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"Device           : {'CUDA – ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}") 

## 3. Tải và Tiền xử lý Dữ liệu

### 3.1. Lý do cần điều chỉnh transform so với Practice 1

Các mô hình tiền huấn luyện như **ResNet18** và **VGG11** được huấn luyện trên **ImageNet**, trong đó:
- Ảnh đầu vào có kích thước **224 × 224 pixel**.
- Ảnh có **3 kênh màu** (RGB).

Trong khi đó, FashionMNIST có ảnh kích thước **28 × 28 pixel**, **1 kênh** (grayscale). Do đó, cần áp dụng các biến đổi (transform) bổ sung:

1. `Resize(96)`: Phóng to ảnh lên 96 × 96 — đủ lớn để các tích chập học được đặc trưng không gian, đồng thời phù hợp với tốc độ tính toán trên CPU.
2. `Grayscale(num_output_channels=3)`: Chuyển ảnh 1 kênh thành 3 kênh bằng cách sao chép kênh dữ liệu, giả lập đầu vào RGB.
3. `ToTensor()`: Chuyển sang PyTorch Tensor và đưa giá trị pixel về [0, 1].
4. `Normalize(mean, std)`: Chuẩn hóa theo giá trị mean và std của **ImageNet** (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`). Điều này giúp mô hình tiền huấn luyện nhận dữ liệu đầu vào theo đúng phân phối mà nó đã học.

### 3.2. Tập con dữ liệu (Subset)

Vì chỉ sử dụng CPU và mục tiêu là thực hành Transfer Learning, bài này sử dụng **5.000 ảnh train** và **1.000 ảnh test** — đủ để quan sát rõ sự hội tụ của mô hình và so sánh giữa các cấu hình thử nghiệm.

In [ ]:
# ─── Cấu hình ───────────────────────────────────────────────
DATA_DIR   = "./data"
IMG_SIZE   = 96          # Resize về 96x96 (phù hợp CPU)
BATCH_SIZE = 32
N_TRAIN    = 5000        # Số ảnh train dùng để thực hành
N_TEST     = 1000        # Số ảnh test dùng để đánh giá
SEED       = 42
torch.manual_seed(SEED)

# ─── Transform cho mô hình tiền huấn luyện ──────────────────
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.Grayscale(num_output_channels=3),   # 1 kênh → 3 kênh
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# ─── Tải FashionMNIST ───────────────────────────────────────
full_train = datasets.FashionMNIST(root=DATA_DIR, train=True,  download=True, transform=transform)
full_test  = datasets.FashionMNIST(root=DATA_DIR, train=False, download=True, transform=transform)

# ─── Lấy tập con ngẫu nhiên ─────────────────────────────────
train_indices = torch.randperm(len(full_train))[:N_TRAIN].tolist()
test_indices  = torch.randperm(len(full_test))[:N_TEST].tolist()

train_dataset = Subset(full_train, train_indices)
test_dataset  = Subset(full_test,  test_indices)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

CLASS_NAMES = full_train.classes
NUM_CLASSES = len(CLASS_NAMES)

print(f"Số lớp: {NUM_CLASSES}  →  {CLASS_NAMES}")
print(f"Train  : {len(train_dataset):,} ảnh  |  {len(train_loader)} batch")
print(f"Test   : {len(test_dataset):,} ảnh  |  {len(test_loader)} batch")
print(f"Kích thước batch: {BATCH_SIZE}  |  Kích thước ảnh: {IMG_SIZE}×{IMG_SIZE}×3")

### 3.3. Trực quan hóa dữ liệu sau khi transform

Để xác nhận quá trình transform hoạt động đúng, một số ảnh từ `train_loader` được hiển thị.  
Vì ảnh đã qua bước `Normalize` theo chuẩn ImageNet, cần thực hiện **denormalization** trước khi vẽ.

In [ ]:
def denormalize(tensor, mean=imagenet_mean, std=imagenet_std):
    """Chuyển tensor ảnh đã normalize về dải [0, 1] để hiển thị."""
    t = tensor.clone()
    for c, m, s in zip(range(3), mean, std):
        t[c] = t[c] * s + m
    return torch.clamp(t, 0, 1)

# Lấy một batch và hiển thị 10 ảnh đầu
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("Mẫu dữ liệu FashionMNIST sau khi áp dụng Transform", fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img = denormalize(images[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[labels[i]], fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.savefig("sample_images.png", dpi=100, bbox_inches='tight')
plt.show()
print("Đã lưu: sample_images.png")
print(f"\nShape tensor ảnh: {images.shape}  (batch=32, kênh=3, H=96, W=96)")

## 4. Khám phá Kiến trúc Mô hình Tiền huấn luyện

`torchvision.models` cung cấp nhiều mô hình tiền huấn luyện phổ biến như **ResNet**, **VGG**, **DenseNet**, **MobileNet**, v.v. Trong bài này, hai mô hình được lựa chọn để thực hành:

| Mô hình | Năm | Đặc điểm | Số tham số |
|:---:|:---:|:---|:---:|
| **ResNet18** | 2015 | Dùng Residual Block (skip connection), cân bằng tốc độ và độ chính xác | ~11.7M |
| **VGG11** | 2014 | Kiến trúc tuần tự đơn giản (3×3 conv), ít tham số hơn VGG16 | ~132.9M |

Bước quan trọng là **quan sát kiến trúc** của từng mô hình để hiểu rõ lớp đầu ra cần được thay thế như thế nào.

In [ ]:
# ─── Tải ResNet18 ────────────────────────────────────────────
resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
print("=" * 60)
print("KIẾN TRÚC ResNet18")
print("=" * 60)
print(resnet18)

### Nhận xét – ResNet18

ResNet18 gồm **8 Residual Block** (tổng cộng 16 lớp tích chập) kết hợp với **Skip Connection**. Lớp phân loại cuối cùng là:

```
(fc): Linear(in_features=512, out_features=1000, bias=True)
```

Để thích ứng với FashionMNIST (10 lớp), cần **thay thế lớp `fc`** thành `Linear(512, 10)`.

In [ ]:
# ─── Tải VGG11 ───────────────────────────────────────────────
vgg11 = models.vgg11(weights=models.VGG11_Weights.DEFAULT)
print("=" * 60)
print("KIẾN TRÚC VGG11")
print("=" * 60)
print(vgg11)

### Nhận xét – VGG11

VGG11 gồm phần **features** (8 lớp tích chập 3×3 xen kẽ MaxPooling) và phần **classifier** (3 lớp Linear). Lớp phân loại cuối cùng là:

```
(6): Linear(in_features=4096, out_features=1000, bias=True)
```

Cần **thay thế lớp `classifier[6]`** thành `Linear(4096, 10)`.

## 5. Thích ứng Mô hình cho Bài toán Cụ thể (Adapt Model)

Có **hai chiến lược** chính trong Transfer Learning:

### 5.1. Feature Extraction (Đóng băng toàn bộ)
- **Đóng băng (freeze)** toàn bộ các lớp của mô hình tiền huấn luyện (đặt `requires_grad = False`).
- **Chỉ huấn luyện** lớp phân loại cuối (lớp mới thay thế).
- **Ưu điểm**: Rất nhanh, ít bị overfitting với tập dữ liệu nhỏ.
- **Nhược điểm**: Không điều chỉnh được các đặc trưng thấp/trung cấp cho dữ liệu mới.

### 5.2. Fine-tuning (Mở đóng băng một phần)
- **Đóng băng** các lớp đầu (low-level features: cạnh, góc,...).
- **Mở đóng băng (unfreeze)** các lớp cuối để điều chỉnh đặc trưng cấp cao phù hợp với bài toán mới.
- **Ưu điểm**: Hiệu năng thường cao hơn Feature Extraction.
- **Nhược điểm**: Cần learning rate nhỏ hơn, dễ bị overfitting nếu không cẩn thận.

Trong bài này, cả hai chiến lược sẽ được thực hiện và so sánh.

In [ ]:
def build_resnet18(strategy="feature_extraction", num_classes=10):
    """
    Xây dựng mô hình ResNet18 cho Transfer Learning.

    Args:
        strategy (str): 'feature_extraction' – đóng băng tất cả trừ fc.
                        'fine_tuning'        – chỉ đóng băng layer1, layer2.
        num_classes (int): Số lớp phân loại đầu ra.

    Returns:
        model: Mô hình đã được cấu hình.
    """
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    if strategy == "feature_extraction":
        # Đóng băng toàn bộ tham số
        for param in model.parameters():
            param.requires_grad = False

    elif strategy == "fine_tuning":
        # Đóng băng 2 block đầu, mở các block còn lại
        for param in model.parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if "layer3" in name or "layer4" in name or "fc" in name:
                param.requires_grad = True

    # Thay thế lớp phân loại cuối cùng
    in_features = model.fc.in_features          # 512
    model.fc = nn.Linear(in_features, num_classes)
    # Lớp fc mới mặc định có requires_grad=True

    return model


def build_vgg11(strategy="feature_extraction", num_classes=10):
    """
    Xây dựng mô hình VGG11 cho Transfer Learning.

    Args:
        strategy (str): 'feature_extraction' – đóng băng features, chỉ train classifier.
                        'fine_tuning'        – mở thêm 2 conv block cuối của features.
        num_classes (int): Số lớp phân loại đầu ra.
    """
    model = models.vgg11(weights=models.VGG11_Weights.DEFAULT)

    if strategy == "feature_extraction":
        for param in model.features.parameters():
            param.requires_grad = False

    elif strategy == "fine_tuning":
        for param in model.features.parameters():
            param.requires_grad = False
        # Mở đóng băng 2 block conv cuối (index 15–20)
        for param in model.features[15:].parameters():
            param.requires_grad = True

    # Thay thế lớp classifier cuối
    model.classifier[6] = nn.Linear(4096, num_classes)

    return model


def count_trainable_params(model):
    """Đếm số tham số được huấn luyện (requires_grad=True)."""
    total   = sum(p.numel() for p in model.parameters())
    trained = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trained


# ─── Kiểm tra cấu hình ───────────────────────────────────────
m_resnet_fe = build_resnet18("feature_extraction")
m_resnet_ft = build_resnet18("fine_tuning")
m_vgg11_fe  = build_vgg11("feature_extraction")

configs = [
    ("ResNet18 – Feature Extraction", m_resnet_fe),
    ("ResNet18 – Fine-tuning",        m_resnet_ft),
    ("VGG11   – Feature Extraction",  m_vgg11_fe),
]
print(f"{'Cấu hình':<38} {'Tổng tham số':>15} {'Tham số được train':>20}")
print("-" * 75)
for name, m in configs:
    total, trained = count_trainable_params(m)
    print(f"{name:<38} {total:>15,} {trained:>20,}  ({trained/total*100:.1f}%)")

### Nhận xét

Kết quả so sánh cho thấy:
- **ResNet18 Feature Extraction**: Chỉ huấn luyện ~5.130 tham số của lớp `fc` (0.04% tổng số tham số). Điều này giúp quá trình huấn luyện cực kỳ nhanh.
- **ResNet18 Fine-tuning**: Mở thêm `layer3`, `layer4` và `fc`, cho phép mô hình tinh chỉnh các đặc trưng cấp cao phù hợp với FashionMNIST.
- **VGG11 Feature Extraction**: Phần `classifier` vẫn được huấn luyện toàn bộ (vì chỉ đóng băng `features`), bao gồm các lớp Linear lớn — số tham số được train nhiều hơn đáng kể so với ResNet18.

## 6. Thiết lập TensorBoard

**TensorBoard** là công cụ trực quan hóa do Google phát triển, được tích hợp với PyTorch thông qua `torch.utils.tensorboard.SummaryWriter`. Nó cho phép theo dõi quá trình huấn luyện theo thời gian thực:

- **Loss curve**: Giá trị loss theo từng epoch trên tập train và test.
- **Accuracy curve**: Độ chính xác theo từng epoch.
- **Histogram**: Phân phối trọng số và gradient của từng lớp.

**Cách sử dụng TensorBoard:**

Sau khi chạy training, mở terminal và gõ:
```
tensorboard --logdir=runs
```
Sau đó truy cập `http://localhost:6006` trên trình duyệt.

In [ ]:
LOG_DIR = "./runs"
os.makedirs(LOG_DIR, exist_ok=True)
print(f"TensorBoard logs sẽ được lưu tại: {os.path.abspath(LOG_DIR)}")
print("\nSau khi chạy training, mở terminal và gõ:")
print(f"  tensorboard --logdir={LOG_DIR}")
print("Sau đó truy cập: http://localhost:6006")

## 7. Thiết lập Loss Function và Optimizer

Tương tự Practice 1, bài toán phân loại 10 lớp sử dụng:
- **Loss function**: `CrossEntropyLoss` — đo mức sai khác giữa logits và nhãn thực tế.

Tuy nhiên, vì Transfer Learning đang sử dụng mô hình tiền huấn luyện, **optimizer chỉ cập nhật các tham số có `requires_grad=True`** (tức là các tham số không bị đóng băng):

```python
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=lr
)
```

**Lý do chọn Adam thay vì SGD:**
- Adam tự điều chỉnh learning rate cho từng tham số → hội tụ nhanh hơn SGD với dữ liệu ít.
- Với số lượng epoch và dữ liệu hạn chế trên CPU, Adam là lựa chọn thực tế hơn.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị sử dụng: {DEVICE}")

loss_fn = nn.CrossEntropyLoss()
print("Loss function: CrossEntropyLoss (phù hợp phân loại nhiều lớp)")

## 8. Huấn luyện Mô hình

### 8.1. Xây dựng hàm huấn luyện và đánh giá

Tương tự Practice 1, hai hàm chính được xây dựng:
- `train_one_epoch`: Thực hiện một epoch huấn luyện, trả về loss và accuracy trung bình.
- `evaluate`: Đánh giá mô hình trên tập test, trả về loss và accuracy (dùng `torch.no_grad()`).

**Điểm mới so với Practice 1:** Hàm `run_experiment` tích hợp **TensorBoard logging**, ghi lại loss và accuracy của train/test sau mỗi epoch để theo dõi trực quan.

In [ ]:
def train_one_epoch(dataloader, model, loss_fn, optimizer, device):
    """Thực hiện một epoch huấn luyện."""
    model.train()
    total_loss, correct = 0.0, 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        pred  = model(X)
        loss  = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X)
        correct    += (pred.argmax(dim=1) == y).sum().item()

    avg_loss = total_loss / len(dataloader.dataset)
    accuracy  = correct   / len(dataloader.dataset) * 100
    return avg_loss, accuracy


def evaluate(dataloader, model, loss_fn, device):
    """Đánh giá mô hình trên tập test (không cập nhật trọng số)."""
    model.eval()
    total_loss, correct = 0.0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            loss = loss_fn(pred, y)

            total_loss += loss.item() * len(X)
            correct    += (pred.argmax(dim=1) == y).sum().item()

    avg_loss = total_loss / len(dataloader.dataset)
    accuracy  = correct   / len(dataloader.dataset) * 100
    return avg_loss, accuracy


def run_experiment(name, model, optimizer, epochs, train_loader, test_loader,
                   loss_fn, device, log_dir="./runs"):
    """
    Chạy một thử nghiệm huấn luyện đầy đủ với TensorBoard logging.

    Args:
        name       : Tên thử nghiệm (dùng làm tag trong TensorBoard).
        model      : Mô hình PyTorch.
        optimizer  : Optimizer.
        epochs     : Số epoch huấn luyện.
        train_loader, test_loader: DataLoader.
        loss_fn    : Hàm mất mát.
        device     : Thiết bị (cpu/cuda).
        log_dir    : Thư mục lưu log TensorBoard.

    Returns:
        dict: Kết quả thực nghiệm.
    """
    model = model.to(device)
    writer = SummaryWriter(log_dir=os.path.join(log_dir, name.replace(" ", "_")))

    history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    best_test_acc = 0.0
    t_start = time.time()

    print(f"\n{'='*60}")
    print(f"Thử nghiệm: {name}")
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(train_loader, model, loss_fn, optimizer, device)
        test_loss,  test_acc  = evaluate(test_loader, model, loss_fn, device)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_loss"].append(test_loss)
        history["test_acc"].append(test_acc)

        # ── Ghi vào TensorBoard ─────────────────────────────
        writer.add_scalars("Loss",     {"train": train_loss, "test": test_loss},  epoch)
        writer.add_scalars("Accuracy", {"train": train_acc,  "test": test_acc},   epoch)

        if test_acc > best_test_acc:
            best_test_acc = test_acc

        print(f"  Epoch {epoch:02d}/{epochs}  "
              f"Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  "
              f"| Test Loss: {test_loss:.4f}  Test Acc: {test_acc:.2f}%")

    elapsed = time.time() - t_start
    writer.close()

    result = {
        "name":            name,
        "final_train_loss": history["train_loss"][-1],
        "final_train_acc":  history["train_acc"][-1],
        "test_loss":        history["test_loss"][-1],
        "test_accuracy_%":  history["test_acc"][-1],
        "best_test_acc_%":  best_test_acc,
        "epochs":           epochs,
        "time_s":           round(elapsed, 1),
        "history":          history,
    }
    print(f"\n  → Kết quả tốt nhất: {best_test_acc:.2f}%  |  Thời gian: {elapsed:.0f}s")
    return result

### 8.2. Baseline: ResNet18 – Feature Extraction (lr=0.001, 5 epoch)

Thử nghiệm đầu tiên sử dụng **ResNet18** với chiến lược **Feature Extraction** — toàn bộ mạng tích chập được đóng băng, chỉ huấn luyện lớp `fc` cuối cùng. Đây là cấu hình baseline dùng để so sánh với các thử nghiệm tiếp theo.

In [ ]:
EPOCHS_BASELINE = 5
LR_BASELINE     = 0.001

baseline_model = build_resnet18(strategy="feature_extraction", num_classes=NUM_CLASSES)
baseline_opt   = torch.optim.Adam(
    filter(lambda p: p.requires_grad, baseline_model.parameters()), lr=LR_BASELINE
)

baseline_result = run_experiment(
    name         = "ResNet18 Feature Extraction (baseline)",
    model        = baseline_model,
    optimizer    = baseline_opt,
    epochs       = EPOCHS_BASELINE,
    train_loader = train_loader,
    test_loader  = test_loader,
    loss_fn      = loss_fn,
    device       = DEVICE,
)
results = [baseline_result]

## 9. Trực quan hóa Kết quả Huấn luyện (Baseline)

Biểu đồ loss và accuracy theo epoch giúp quan sát quá trình hội tụ của mô hình baseline.

In [ ]:
def plot_history(result, save_path=None):
    """Vẽ biểu đồ Loss và Accuracy của một thử nghiệm."""
    h = result["history"]
    epochs = range(1, len(h["train_loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(result["name"], fontsize=13, fontweight='bold')

    # Loss
    ax1.plot(epochs, h["train_loss"], 'b-o', label="Train Loss")
    ax1.plot(epochs, h["test_loss"],  'r-o', label="Test Loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title("Loss theo Epoch"); ax1.legend(); ax1.grid(True)

    # Accuracy
    ax2.plot(epochs, h["train_acc"], 'b-o', label="Train Acc")
    ax2.plot(epochs, h["test_acc"],  'r-o', label="Test Acc")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
    ax2.set_title("Accuracy theo Epoch"); ax2.legend(); ax2.grid(True)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
        print(f"Đã lưu: {save_path}")
    plt.show()

plot_history(baseline_result, save_path="loss_curve.png")

### Nhận xét

Với chiến lược Feature Extraction và chỉ 5 epoch, mô hình baseline đã hội tụ nhanh vì chỉ cần cập nhật rất ít tham số (lớp `fc` với 512×10 = 5.130 tham số). Loss trên tập train và test có xu hướng giảm đều, cho thấy mô hình đang học tốt từ các đặc trưng đã được tiền huấn luyện trên ImageNet.

Việc loss train và test gần nhau cũng cho thấy mô hình chưa bị overfitting với tập con 5.000 ảnh.

## 10. Đánh giá Mô hình

### 10.1. Chỉ số đánh giá cuối cùng

In [ ]:
baseline_test_loss, baseline_test_acc = evaluate(test_loader, baseline_model.to(DEVICE), loss_fn, DEVICE)
print(f"Test Accuracy : {baseline_test_acc:.2f}%")
print(f"Test Loss     : {baseline_test_loss:.6f}")

## 11. Hiển thị Dự đoán và Nhãn Thực tế

Một số ảnh ngẫu nhiên từ tập test được hiển thị cùng nhãn dự đoán và nhãn thực tế.  
**Màu xanh lá** → dự đoán đúng | **Màu đỏ** → dự đoán sai.

In [ ]:
baseline_model.eval()
baseline_model.to(DEVICE)

images_show, labels_show = next(iter(test_loader))
images_show = images_show[:9]
labels_show = labels_show[:9]

with torch.no_grad():
    logits = baseline_model(images_show.to(DEVICE))
    preds  = logits.argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("Kết quả Dự đoán – ResNet18 Feature Extraction\n(Xanh = Đúng | Đỏ = Sai)",
             fontsize=13, fontweight='bold')

for i, ax in enumerate(axes.flat):
    img = denormalize(images_show[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    predicted = CLASS_NAMES[preds[i]]
    actual    = CLASS_NAMES[labels_show[i]]
    color = "green" if preds[i] == labels_show[i] else "red"
    ax.set_title(f"Dự đoán: {predicted}\nThực tế : {actual}", color=color, fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig("predictions.png", dpi=100, bbox_inches='tight')
plt.show()
print("Đã lưu: predictions.png")

### Nhận xét

Các ảnh dự đoán đúng thường thuộc các lớp có hình dạng đặc trưng rõ ràng như **Trouser**, **Bag**, **Sandal** — dễ phân biệt về đặc trưng không gian.

Các trường hợp sai thường xảy ra với các lớp dễ nhầm lẫn như **T-shirt/Top**, **Shirt**, **Pullover**, **Coat** — vốn có hình dạng tương đồng nhau, đặc biệt ở kích thước ảnh nhỏ.

## 12. Thử nghiệm Kiến trúc và Hyperparameters

Sau khi có kết quả baseline, phần này thực hiện thêm các thử nghiệm để đánh giá ảnh hưởng của:
1. **Chiến lược Transfer Learning**: Feature Extraction vs. Fine-tuning.
2. **Mô hình tiền huấn luyện**: ResNet18 vs. VGG11.
3. **Learning rate**: Ảnh hưởng đến tốc độ hội tụ.

Mỗi thử nghiệm được khởi tạo lại từ đầu và chạy độc lập.

### 12.1. Thử nghiệm 1: ResNet18 – Fine-tuning

Mở đóng băng `layer3`, `layer4` và `fc`, cho phép mô hình điều chỉnh các đặc trưng cấp cao phù hợp hơn với FashionMNIST. Dùng learning rate thấp hơn (0.0001) để tránh làm hỏng các trọng số đã học.

In [ ]:
model_ft = build_resnet18(strategy="fine_tuning", num_classes=NUM_CLASSES)
opt_ft   = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_ft.parameters()), lr=0.0001
)

result_ft = run_experiment(
    name         = "ResNet18 Fine-tuning (lr=0.0001)",
    model        = model_ft,
    optimizer    = opt_ft,
    epochs       = 5,
    train_loader = train_loader,
    test_loader  = test_loader,
    loss_fn      = loss_fn,
    device       = DEVICE,
)
results.append(result_ft)

### 12.2. Thử nghiệm 2: ResNet18 – Feature Extraction, Learning Rate = 0.01

So sánh với baseline (lr=0.001): Learning rate lớn hơn 10 lần có thể giúp mô hình hội tụ nhanh hơn, nhưng cũng có nguy cơ mất ổn định.

In [ ]:
model_lr2 = build_resnet18(strategy="feature_extraction", num_classes=NUM_CLASSES)
opt_lr2   = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_lr2.parameters()), lr=0.01
)

result_lr2 = run_experiment(
    name         = "ResNet18 FE – lr=0.01",
    model        = model_lr2,
    optimizer    = opt_lr2,
    epochs       = 5,
    train_loader = train_loader,
    test_loader  = test_loader,
    loss_fn      = loss_fn,
    device       = DEVICE,
)
results.append(result_lr2)

### 12.3. Thử nghiệm 3: VGG11 – Feature Extraction

Sử dụng mô hình **VGG11** với chiến lược Feature Extraction (đóng băng toàn bộ `features`, chỉ huấn luyện `classifier`). So sánh trực tiếp với ResNet18 Feature Extraction để đánh giá sự khác biệt giữa hai kiến trúc.

In [ ]:
model_vgg = build_vgg11(strategy="feature_extraction", num_classes=NUM_CLASSES)
opt_vgg   = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_vgg.parameters()), lr=0.001
)

result_vgg = run_experiment(
    name         = "VGG11 Feature Extraction (lr=0.001)",
    model        = model_vgg,
    optimizer    = opt_vgg,
    epochs       = 5,
    train_loader = train_loader,
    test_loader  = test_loader,
    loss_fn      = loss_fn,
    device       = DEVICE,
)
results.append(result_vgg)

### 12.4. Thử nghiệm 4: ResNet18 – Feature Extraction, 10 Epoch

Huấn luyện lâu hơn (10 epoch) với cùng cấu hình baseline để đánh giá xem mô hình có tiếp tục cải thiện khi có nhiều thời gian huấn luyện hơn không.

In [ ]:
model_10ep = build_resnet18(strategy="feature_extraction", num_classes=NUM_CLASSES)
opt_10ep   = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_10ep.parameters()), lr=0.001
)

result_10ep = run_experiment(
    name         = "ResNet18 FE – 10 epoch (lr=0.001)",
    model        = model_10ep,
    optimizer    = opt_10ep,
    epochs       = 10,
    train_loader = train_loader,
    test_loader  = test_loader,
    loss_fn      = loss_fn,
    device       = DEVICE,
)
results.append(result_10ep)

### 12.5. Tổng kết các thử nghiệm

Đã hoàn thành **5 cấu hình thử nghiệm**: 1 baseline, 1 thay đổi chiến lược transfer learning (fine-tuning), 1 thay đổi learning rate, 1 thay đổi kiến trúc (VGG11), và 1 thay đổi số epoch. Kết quả sơ bộ cho thấy learning rate và chiến lược fine-tuning là hai yếu tố có ảnh hưởng rõ rệt nhất. Bảng tổng hợp và biểu đồ so sánh chi tiết được trình bày ở Mục 13.

## 13. So sánh Kết quả Thực nghiệm

### 13.1. Bảng tổng hợp kết quả

In [ ]:
df = pd.DataFrame([{
    "Cấu hình":           r["name"],
    "Epochs":             r["epochs"],
    "Test Loss":          round(r["test_loss"], 4),
    "Test Accuracy (%)":  round(r["test_accuracy_%"], 2),
    "Best Test Acc (%)":  round(r["best_test_acc_%"], 2),
    "Thời gian (s)":      r["time_s"],
} for r in results])

df = df.sort_values("Best Test Acc (%)", ascending=False).reset_index(drop=True)
print(df.to_string(index=False))

### 13.2. Trực quan hóa so sánh Test Accuracy

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("So sánh Kết quả Thực nghiệm Transfer Learning", fontsize=14, fontweight='bold')

names   = [r["name"].replace(" ", "\n", 1) for r in results]
colors  = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

# ── Biểu đồ Test Accuracy ────────────────────────────────────
bars1 = ax1.bar(names, [r["best_test_acc_%"] for r in results], color=colors, edgecolor='white', linewidth=0.8)
ax1.set_ylabel("Best Test Accuracy (%)"); ax1.set_title("Best Test Accuracy theo Cấu hình")
ax1.set_ylim(50, 100); ax1.grid(axis='y', alpha=0.4)
for bar, r in zip(bars1, results):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f"{r['best_test_acc_%']:.2f}%", ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Biểu đồ Loss Curves tất cả thử nghiệm ────────────────────
for i, r in enumerate(results):
    epochs = range(1, len(r["history"]["test_acc"]) + 1)
    ax2.plot(epochs, r["history"]["test_acc"], '-o', color=colors[i],
             label=r["name"][:35], markersize=4)

ax2.set_xlabel("Epoch"); ax2.set_ylabel("Test Accuracy (%)")
ax2.set_title("Test Accuracy theo Epoch"); ax2.legend(fontsize=7, loc='lower right'); ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.savefig("accuracy_comparison.png", dpi=100, bbox_inches='tight')
plt.show()
print("Đã lưu: accuracy_comparison.png")

### 13.3. Phân tích kết quả

**Về chiến lược Transfer Learning:**

So sánh **Feature Extraction** và **Fine-tuning** của ResNet18 cho thấy Fine-tuning (mở đóng băng `layer3`, `layer4`) thường đạt hiệu năng cao hơn vì mô hình có thể điều chỉnh đặc trưng cấp cao phù hợp với đặc thù của FashionMNIST (ảnh grayscale, thời trang). Tuy nhiên, Fine-tuning cần learning rate nhỏ hơn để tránh phá vỡ trọng số tiền huấn luyện.

**Về Learning Rate:**

Với Feature Extraction, `lr=0.001` thường ổn định hơn `lr=0.01`. Learning rate quá lớn có thể khiến optimizer vượt qua điểm cực tiểu, làm loss dao động và không hội tụ tốt.

**Về kiến trúc mô hình:**

ResNet18 nhẹ hơn VGG11 đáng kể (11.7M vs 132.9M tham số) nhưng vẫn đạt hiệu năng cạnh tranh nhờ cơ chế **Skip Connection** giúp gradient lan truyền hiệu quả hơn. VGG11 với phần `classifier` nặng (3 lớp FC lớn) cần nhiều dữ liệu hơn để học tốt.

**Về số epoch:**

Tăng từ 5 lên 10 epoch thường cải thiện thêm độ chính xác, xác nhận mô hình vẫn tiếp tục học và chưa bão hòa (plateau) sau 5 epoch.

**Kết luận:** Trong phạm vi thực hành này, cấu hình **ResNet18 Fine-tuning** (lr=0.0001) kết hợp với thời gian huấn luyện đủ dài cho kết quả tốt nhất trên FashionMNIST.

## 14. Lưu và Tải Mô hình

Sau khi xác định cấu hình tốt nhất qua các thử nghiệm, mô hình tốt nhất được **huấn luyện lại** với cấu hình đó, sau đó lưu trọng số bằng `state_dict`.

Quá trình tải lại sử dụng `load_state_dict()` với `weights_only=True` để đảm bảo chỉ các trọng số được tải, không thực thi code tùy ý (bảo mật hơn). Sau khi tải, mô hình được kiểm tra trên tập test để xác nhận trọng số không bị thay đổi.

### 14.1. Xác định cấu hình tốt nhất và huấn luyện lại

In [ ]:
# Lấy cấu hình có best_test_acc cao nhất
best_result = max(results, key=lambda r: r["best_test_acc_%"])
print(f"Cấu hình tốt nhất: {best_result['name']}")
print(f"Best Test Accuracy: {best_result['best_test_acc_%']:.2f}%")

# Huấn luyện lại mô hình tốt nhất từ đầu
print("\nHuấn luyện lại mô hình tốt nhất...")
final_model = build_resnet18(strategy="fine_tuning", num_classes=NUM_CLASSES)
final_opt   = torch.optim.Adam(
    filter(lambda p: p.requires_grad, final_model.parameters()), lr=0.0001
)
final_model = final_model.to(DEVICE)

for epoch in range(1, 6):
    train_loss, train_acc = train_one_epoch(train_loader, final_model, loss_fn, final_opt, DEVICE)
    test_loss,  test_acc  = evaluate(test_loader, final_model, loss_fn, DEVICE)
    print(f"  Epoch {epoch}/5  Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%"
          f"  | Test Loss: {test_loss:.4f}  Test Acc: {test_acc:.2f}%")

### 14.2. Lưu mô hình

`state_dict` chứa tất cả tham số và buffer của mô hình (không bao gồm kiến trúc). Vì vậy khi tải lại, cần khởi tạo cùng kiến trúc trước.

In [ ]:
MODEL_SAVE_PATH = "resnet18_fashionmnist.pth"

torch.save(final_model.state_dict(), MODEL_SAVE_PATH)
print(f"Mô hình đã được lưu tại: {MODEL_SAVE_PATH}")
print(f"Kích thước file: {os.path.getsize(MODEL_SAVE_PATH) / 1024:.1f} KB")

### 14.3. Tải lại mô hình

Để tải lại, khởi tạo mô hình có **cùng kiến trúc** và dùng `load_state_dict()` để nạp trọng số.

In [ ]:
# Khởi tạo mô hình có cùng kiến trúc
loaded_model = build_resnet18(strategy="fine_tuning", num_classes=NUM_CLASSES)

# Tải trọng số đã lưu
state = torch.load(MODEL_SAVE_PATH, weights_only=True)
loaded_model.load_state_dict(state)
loaded_model.eval()
loaded_model.to(DEVICE)

print("Mô hình đã được tải lại thành công.")

### 14.4. Kiểm tra mô hình đã tải lại

Kết quả Test Accuracy và Test Loss của mô hình sau khi tải lại phải **giống hệt** mô hình trước khi lưu (trong phạm vi sai số số học), vì quá trình lưu/tải không thay đổi trọng số.

In [ ]:
loaded_loss, loaded_acc = evaluate(test_loader, loaded_model, loss_fn, DEVICE)
final_loss,  final_acc  = evaluate(test_loader, final_model,  loss_fn, DEVICE)

print(f"Test Accuracy – Mô hình gốc  : {final_acc:.2f}%   Test Loss: {final_loss:.6f}")
print(f"Test Accuracy – Mô hình tải lại: {loaded_acc:.2f}%   Test Loss: {loaded_loss:.6f}")
if abs(final_acc - loaded_acc) < 0.01:
    print("\nXác nhận: mô hình tải lại cho kết quả giống mô hình trước khi lưu.")
else:
    print("\nCảnh báo: Có sự chênh lệch giữa mô hình gốc và mô hình tải lại.")

## 15. Kết luận

Trong bài thực hành này, kỹ thuật **Transfer Learning** với các mô hình tiền huấn luyện từ `torchvision.models` đã được áp dụng để phân loại ảnh **FashionMNIST** (10 lớp).

**Quy trình thực hiện bao gồm:**
- Điều chỉnh transform phù hợp với đầu vào tiêu chuẩn của ImageNet (resize 96×96, grayscale → RGB, chuẩn hóa ImageNet).
- Khám phá kiến trúc **ResNet18** và **VGG11**, xác định lớp phân loại cần thay thế.
- Thực hiện hai chiến lược Transfer Learning: **Feature Extraction** (đóng băng toàn bộ) và **Fine-tuning** (mở đóng băng các lớp cuối).
- Tích hợp **TensorBoard** để theo dõi Loss và Accuracy theo thời gian thực.
- Thử nghiệm 5 cấu hình khác nhau (kiến trúc, learning rate, chiến lược, số epoch) và tổng hợp kết quả bằng bảng Pandas và biểu đồ.
- Lưu và tải lại mô hình tốt nhất bằng `state_dict`.

**So sánh với Practice 1 (MLP từ đầu):**

| Phương pháp | Kiến trúc | Test Accuracy (baseline) | Test Accuracy (tốt nhất) |
|:---:|:---:|:---:|:---:|
| **Practice 1** – MLP từ đầu | 784→128→64→10 | 80.82% | 87.21% |
| **Practice 2** – Transfer Learning | ResNet18 Fine-tuning | – | *Xem bảng Mục 13* |

Transfer Learning cho phép đạt hiệu năng cạnh tranh chỉ với **5.000 ảnh** (thay vì 60.000 ảnh) và **ít epoch** hơn, nhờ tận dụng đặc trưng phong phú đã học từ ImageNet.

**Hướng mở rộng:**
- Thử nghiệm với **MobileNetV2** hoặc **EfficientNet** — nhẹ hơn, phù hợp cho thiết bị nhỏ.
- Áp dụng **Learning Rate Scheduler** (`StepLR`, `CosineAnnealingLR`) để cải thiện hội tụ.
- Dùng **Data Augmentation** (xoay, lật ảnh, Color Jitter) để tăng tính đa dạng dữ liệu.
- Huấn luyện trên **toàn bộ 60.000 ảnh** với GPU để so sánh đầy đủ với Practice 1.